# Linear Regression: A to Z Masterclass

Welcome to this comprehensive guide on Linear Regression! In this notebook, we'll cover everything from the mathematical foundations to advanced implementation using `scikit-learn`, complete with visualizations and best practices.

## Table of Contents
1. What is Linear Regression?
2. Assumptions of Linear Regression
3. Simple Linear Regression (Theory & Implementation)
4. Multiple Linear Regression
5. Model Evaluation Metrics
6. Dealing with Non-Linearity (Polynomial Regression)
7. Regularization (Ridge & Lasso)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set plot style
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## 1. What is Linear Regression?

Linear Regression is a supervised machine learning algorithm used for predicting a **continuous target variable** based on one or more independent predictor variables.

**The Goal:** Find the best-fitting straight line (or hyperplane in higher dimensions) that describes the relationship between the predictors ($X$) and the target ($y$).

**Equation of a Line:**
$$ y = \beta_0 + \beta_1 X_1 + \beta_2 X_2 + \dots + \beta_n X_n + \epsilon $$
- $y$: Predicted value (Dependent variable)
- $\beta_0$: Intercept (Bias term)
- $\beta_1, \dots, \beta_n$: Coefficients (Weights)
- $X_1, \dots, X_n$: Features (Independent variables)
- $\epsilon$: Error term


## 2. Assumptions of Linear Regression
Before applying linear regression, we must ensure our data meets these assumptions (LINE):

1. **L**inearity: The relationship between X and y is linear.
2. **I**ndependence: Observations are independent of each other.
3. **N**ormality of Residuals: The residuals (errors) are normally distributed.
4. **E**qual Variance (Homoscedasticity): The residuals have constant variance across all levels of X.


## 3. Simple Linear Regression

Let's start with Simple Linear Regression, which involves only one feature. We'll generate some synthetic sample data to visualize it perfectly.


In [ ]:
# Generating synthetic sample data for Simple Linear Regression
np.random.seed(42)
X_simple = 2 * np.random.rand(100, 1)
y_simple = 4 + 3 * X_simple + np.random.randn(100, 1)

plt.scatter(X_simple, y_simple, color='blue', alpha=0.6)
plt.title('Synthetic Data for Simple Linear Regression')
plt.xlabel('X (Feature)')
plt.ylabel('y (Target)')
plt.show()


### Training the Model
We will use `scikit-learn` to fit a Linear Regression model to our data.


In [ ]:
# 1. Initialize the model
simple_model = LinearRegression()

# 2. Fit the model to the data
simple_model.fit(X_simple, y_simple)

# 3. Extract the learned parameters
intercept = simple_model.intercept_[0]
coefficient = simple_model.coef_[0][0]

print(f"Intercept (beta_0): {intercept:.4f}")
print(f"Coefficient (beta_1): {coefficient:.4f}")
print(f"True Equation: y = 4 + 3 * X")
print(f"Learned Equation: y = {intercept:.4f} + {coefficient:.4f} * X")


### Visualizing the Best Fit Line


In [ ]:
# Make predictions across the range of X
X_new = np.array([[0], [2]])
y_predict = simple_model.predict(X_new)

plt.scatter(X_simple, y_simple, color='blue', alpha=0.6, label='Actual Data')
plt.plot(X_new, y_predict, color='red', linewidth=3, label='Best Fit Line')
plt.title('Simple Linear Regression Fit')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()


## 4. Multiple Linear Regression & 5. Model Evaluation Metrics

Now, let's look at multiple features using the California Housing dataset. We will predict the median house value based on various features.


In [ ]:
from sklearn.datasets import fetch_california_housing

# Load dataset
housing = fetch_california_housing()

# Create a DataFrame
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['Target_MedHouseVal'] = housing.target

display(df.head())

# Let's visualize the correlation matrix to see how features relate to the target
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm', fmt='.2f', vmin=-1, vmax=1)
plt.title('Correlation Matrix')
plt.show()


### Train-Test Split
It is crucial to split our data to evaluate how our model performs on unseen data.


In [ ]:
X = df.drop('Target_MedHouseVal', axis=1)
y = df['Target_MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")


In [ ]:
# Train the Multiple Linear Regression model
multi_model = LinearRegression()
multi_model.fit(X_train, y_train)

# Predict on the test set
y_pred = multi_model.predict(X_test)


### Evaluation Metrics
- **MAE (Mean Absolute Error):** Average of absolute errors. Robust to outliers.
- **MSE (Mean Squared Error):** Average of squared errors. Penalizes large errors.
- **RMSE (Root Mean Squared Error):** Square root of MSE. In the same units as the target.
- **R-squared ($R^2$):** Proportion of variance in the dependent variable explained by the independent variables. (1 is perfect, 0 is baseline).


In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Metrics:")
print("-" * 25)
print(f"MAE:  {mae:.4f}")
print(f"MSE:  {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R2:   {r2:.4f}")


### Visualizing Predictions vs Actuals
A good model will have points closely clustered around the diagonal line.


In [ ]:
plt.scatter(y_test, y_pred, alpha=0.3, color='green')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', lw=2)
plt.xlabel('Actual Median House Value')
plt.ylabel('Predicted Median House Value')
plt.title('Actual vs Predicted')
plt.show()


### Residual Analysis
Let's check the residuals to see if they are normally distributed (Assumption 3) and homoscedastic (Assumption 4).


In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot 1: Residuals distribution
sns.histplot(residuals, kde=True, ax=axes[0], color='purple')
axes[0].set_title('Distribution of Residuals')
axes[0].set_xlabel('Residuals (Actual - Predicted)')

# Plot 2: Residuals vs Predicted (Homoscedasticity check)
axes[1].scatter(y_pred, residuals, alpha=0.3, color='teal')
axes[1].axhline(y=0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted Values')
axes[1].set_xlabel('Predicted Values')
axes[1].set_ylabel('Residuals')

plt.tight_layout()
plt.show()


## 6. Polynomial Regression
Sometimes a straight line doesn't fit the data well. We can add polynomial features ($X^2$, $X^3$, etc.) to capture non-linear relationships while still using a linear model!


In [ ]:
from sklearn.preprocessing import PolynomialFeatures

# Generate non-linear data
np.random.seed(42)
X_poly = 6 * np.random.rand(100, 1) - 3
y_poly = 0.5 * X_poly**2 + X_poly + 2 + np.random.randn(100, 1)

# Create polynomial features (degree 2)
poly_features = PolynomialFeatures(degree=2, include_bias=False)
X_poly_transformed = poly_features.fit_transform(X_poly)

# Fit Linear Regression on polynomial features
poly_model = LinearRegression()
poly_model.fit(X_poly_transformed, y_poly)

# Visualize
X_plot = np.linspace(-3, 3, 100).reshape(100, 1)
X_plot_poly = poly_features.transform(X_plot)
y_plot = poly_model.predict(X_plot_poly)

plt.scatter(X_poly, y_poly, color='blue', alpha=0.5, label='Data')
plt.plot(X_plot, y_plot, color='red', linewidth=2, label='Polynomial Fit (Degree 2)')
plt.title('Polynomial Regression')
plt.xlabel('X')
plt.ylabel('y')
plt.legend()
plt.show()


## 7. Regularization: Ridge & Lasso Regression
When we have many features, our model can **overfit**. Regularization adds a penalty to the loss function to constrain the coefficients.

- **Ridge Regression (L2 Regularization):** Adds penalty equivalent to square of the magnitude of coefficients. Shrinks coefficients towards zero, but rarely exactly to zero.
- **Lasso Regression (L1 Regularization):** Adds penalty equivalent to absolute value of the magnitude of coefficients. Can shrink coefficients exactly to zero, effectively performing **feature selection**.


In [ ]:
from sklearn.linear_model import Ridge, Lasso

# Fit Ridge
ridge_reg = Ridge(alpha=100.0) # alpha is the regularization strength
ridge_reg.fit(X_train, y_train)

# Fit Lasso
lasso_reg = Lasso(alpha=0.1)
lasso_reg.fit(X_train, y_train)

print("Linear Regression R2:", r2_score(y_test, y_pred))
print("Ridge Regression R2: ", ridge_reg.score(X_test, y_test))
print("Lasso Regression R2: ", lasso_reg.score(X_test, y_test))

# Visualizing coefficients
coef_df = pd.DataFrame({
    'Feature': X.columns,
    'Linear': multi_model.coef_,
    'Ridge': ridge_reg.coef_,
    'Lasso': lasso_reg.coef_
})

coef_df.set_index('Feature').plot(kind='bar', figsize=(12, 6))
plt.title('Comparison of Coefficients: Linear vs Ridge vs Lasso')
plt.ylabel('Coefficient Value')
plt.axhline(y=0, color='black', linestyle='-', linewidth=1)
plt.xticks(rotation=45)
plt.show()

print("\nNotice how Lasso has pushed some coefficients exactly to 0!")
